# Solution B1 · A forest on the Titanic driven by the OOB error

**Type**: Implement with scikit-learn

## Objective

Build a pipeline with a `RandomForestClassifier` on the Titanic and use the **out-of-bag** error instead of cross-validation to: see how many trees are needed (OOB curve with `warm_start`), choose `max_features` and `min_samples_leaf`, and compare the OOB estimate with the cross-validation one and with the test set.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def build_prep():
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), ["pclass", "age", "siblings_spouses", "parents_children", "fare"]),
                              ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), ["sex", "port", "deck", "title"])])


def model(**kw):
    return make_pipeline(build_prep(), RandomForestClassifier(oob_score=True, random_state=0, n_jobs=-1, **kw))

## Commented solution

In [ ]:
m = model(warm_start=True)
rows = {}
for n in [5, 10, 20, 50, 100, 200, 400]:
    m.set_params(randomforestclassifier__n_estimators=n).fit(X_tr, y_tr)
    rows[n] = 1 - m[-1].oob_score_
oob_curve = pd.Series(rows)
stable_n = int(next(n for n in oob_curve.index if abs(oob_curve[n] - oob_curve[400]) <= 0.005))
print(oob_curve.round(4).to_string(), "\ntrees to settle the OOB error within 0.005:", stable_n)

In [ ]:
def oob_error(**kw):
    return 1 - model(n_estimators=300, **kw).fit(X_tr, y_tr)[-1].oob_score_


mf_grid = pd.Series({str(mf): oob_error(max_features=mf) for mf in [1, 2, 3, "sqrt", 5, 9]})
leaf_grid = pd.Series({l: oob_error(min_samples_leaf=l) for l in [1, 2, 5, 10]})
best_mf = mf_grid.idxmin()
best_leaf = leaf_grid.idxmin()
print("OOB error by max_features:", mf_grid.round(4).to_dict())
print("OOB error by min_samples_leaf:", leaf_grid.round(4).to_dict())

In [ ]:
mf = int(best_mf) if best_mf.isdigit() else best_mf
final = model(n_estimators=300, max_features=mf, min_samples_leaf=best_leaf).fit(X_tr, y_tr)
oob_err = 1 - final[-1].oob_score_
cv_err = 1 - cross_val_score(model(n_estimators=300, max_features=mf, min_samples_leaf=best_leaf), X_tr, y_tr, cv=cv).mean()
test_err = 1 - final.score(X_te, y_te)
train_acc = final.score(X_tr, y_tr)
print(f"best: max_features={best_mf}, min_samples_leaf={best_leaf}")
print(f"error: OOB {oob_err:.4f} | cross-validation {cv_err:.4f} | test {test_err:.4f} | (training accuracy: {train_acc:.3f})")

## Check

In [ ]:
with verify():
    check(list(oob_curve.index) == [5, 10, 20, 50, 100, 200, 400] and oob_curve[5] > oob_curve[100] + 0.02 and 20 <= stable_n <= 200, "the OOB error drops from 5 to 100 trees by over 2 points and settles between 20 and 200 trees", "warm_start=True and set_params(randomforestclassifier__n_estimators=n)")
    check(len(mf_grid) == 6 and len(leaf_grid) == 4 and str(best_mf) in mf_grid.index and best_leaf in (1, 2, 5, 10), "six values of max_features and four of min_samples_leaf", "one OOB error for each value")
    check(mf_grid["1"] > mf_grid.min() + 0.01, "with max_features=1 the error is worse than the best by over 1 point (trees too weak)", "compare the OOB errors")
    check(0.12 < oob_err < 0.25 and 0.12 < cv_err < 0.25 and 0.12 < test_err < 0.25, "OOB, cross-validation and test errors between 0.12 and 0.25 (accuracies between 75 and 88%)", "1 - accuracy")
    check(abs(oob_err - cv_err) < 0.03 and abs(oob_err - test_err) < 0.05, "the OOB error is close to the cross-validation one (within 3 points) and to the test one (within 5)", "OOB as an honest estimate")
    check(train_acc > 0.85 and 1 - train_acc < oob_err * 0.85, "on the training set the accuracy goes over 0.85: misleading (error clearly below the OOB one)", "model.score(X_tr, y_tr)")
done()

## What you learned

- **The OOB error settles early:** it drops quickly up to about 20 trees (0.27 with 5, 0.22 with 20) and then stays flat (0.223 → 0.214 between 100 and 400 trees, differences of half a point: noise): a number like 200-300 is more than enough. Adding trees never overfits, but after the settling it only costs time and memory.
- **The OOB replaces cross-validation for choosing the hyperparameters:** each value of `max_features` or `min_samples_leaf` requires **one** training, not five. With `max_features=1` the trees are too weak (0.223); the minimum is with 5 features out of 9 (0.205); the best leaf is 5-10 (0.189) against 0.212 with single-example leaves.
- **OOB, cross-validation and test give almost the same number** (error 0.180, 0.184 and 0.183: accuracy 82%): the OOB is an honest estimate of the generalization. **The training accuracy is 0.87**: it is the only one of the four figures never to report.
- The limits of the OOB (correlated or duplicate rows, C1) and the fact that by default it measures accuracy (for AUC, F1 or costs you need to use `oob_decision_function_` as in B3) are the two reasons cross-validation remains the standard for the final comparison.

## Common pitfall

Reporting the `model.score(X_tr, y_tr)` of a random forest: it is almost 1.0 and says nothing. The OOB error (`1 - oob_score_`) is the internal estimate to use when you do not have a validation set.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you choose the hyperparameters of a random forest without cross-validation?</summary>

With the out-of-bag error: each configuration requires a single training and the OOB is an honest estimate (each example is predicted only by the trees that have not seen it). I look at the curve as a function of n_estimators for stability, then search max_features and min_samples_leaf on a small grid.
I use cross-validation or a test set only for the final confirmation and when the rows are not independent.

</details>